# MHIST Preprocessing

Prepare images and metadata for modeling.

In [1]:
import os
import pandas as pd
from PIL import Image

print("Preprocessing notebook started!")

Preprocessing notebook started!


In [ ]:
# Load MHIST annotations
annotations_path = "../data/raw/annotations.csv"

images_candidates = [
    os.path.abspath("../data/raw/images"),
    os.path.abspath("../data/raw/images/images"),
    os.path.abspath("../data/images"),
]
images_path = next((path for path in images_candidates if os.path.exists(path)), images_candidates[0])

if not os.path.exists(images_path):
    raise FileNotFoundError(
        "MHIST image directory not found. Download or extract the dataset into one of: "
        + ", ".join(images_candidates)
    )

df = pd.read_csv(annotations_path)

print("Annotations loaded!")
print("Number of images:", len(df))
print("Columns:", df.columns.tolist())
print("Image directory:", images_path)


Annotations loaded!
Number of images: 3152
Columns: ['Image Name', 'Majority Vote Label', 'Number of Annotators who Selected SSA (Out of 7)', 'Partition']


In [3]:
from sklearn.model_selection import train_test_split

# Separate official training and test data
train_df = df[df["Partition"] == "train"].copy()
test_df = df[df["Partition"] == "test"].copy()

# Create validation set from the official training data
train_df, val_df = train_test_split(
    train_df,
    test_size=0.20,
    stratify=train_df["Majority Vote Label"],
    random_state=42
)

print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))

Training images: 1740
Validation images: 435
Test images: 977


In [4]:
# Check class distribution in each split

print("Training:")
print(train_df["Majority Vote Label"].value_counts())

print("\nValidation:")
print(val_df["Majority Vote Label"].value_counts())

print("\nTest:")
print(test_df["Majority Vote Label"].value_counts())

Training:
Majority Vote Label
HP     1236
SSA     504
Name: count, dtype: int64

Validation:
Majority Vote Label
HP     309
SSA    126
Name: count, dtype: int64

Test:
Majority Vote Label
HP     617
SSA    360
Name: count, dtype: int64


In [5]:
import torch
from torchvision import transforms

# ImageNet normalization values
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

# Training transform
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

# Validation and test transform
val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

print("Image transforms created successfully!")

Image transforms created successfully!


In [6]:
# Test preprocessing on one real MHIST image

first_image_name = train_df["Image Name"].iloc[0]
first_image_path = os.path.join(images_path, first_image_name)

# Open image
image = Image.open(first_image_path).convert("RGB")

# Apply training transform
processed_image = train_transform(image)

print("Original image size:", image.size)
print("Processed image shape:", processed_image.shape)
print("Processed image dtype:", processed_image.dtype)

Original image size: (224, 224)
Processed image shape: torch.Size([3, 224, 224])
Processed image dtype: torch.float32


In [7]:
from torch.utils.data import Dataset

class MHISTDataset(Dataset):

    def __init__(self, dataframe, images_path, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.images_path = images_path
        self.transform = transform

        # Convert class names into numbers
        self.label_map = {
            "HP": 0,
            "SSA": 1
        }

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        # Get image name and label
        image_name = self.dataframe.loc[index, "Image Name"]
        label_name = self.dataframe.loc[index, "Majority Vote Label"]

        # Create full image path
        image_path = os.path.join(self.images_path, image_name)

        # Open image
        image = Image.open(image_path).convert("RGB")

        # Apply preprocessing/augmentation
        if self.transform:
            image = self.transform(image)

        # Convert label to number
        label = self.label_map[label_name]

        return image, label

print("MHISTDataset class created successfully!")

MHISTDataset class created successfully!


In [8]:
# Create Dataset objects

train_dataset = MHISTDataset(
    train_df,
    images_path,
    transform=train_transform
)

val_dataset = MHISTDataset(
    val_df,
    images_path,
    transform=val_test_transform
)

test_dataset = MHISTDataset(
    test_df,
    images_path,
    transform=val_test_transform
)

print("Training dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

Training dataset: 1740
Validation dataset: 435
Test dataset: 977


In [9]:
# Test one sample from the training dataset

image_tensor, label = train_dataset[0]

print("Image tensor shape:", image_tensor.shape)
print("Image tensor dtype:", image_tensor.dtype)
print("Label:", label)

# Convert numeric label back to class name
label_name = "HP" if label == 0 else "SSA"
print("Class:", label_name)

Image tensor shape: torch.Size([3, 224, 224])
Image tensor dtype: torch.float32
Label: 1
Class: SSA


In [10]:
from torch.utils.data import DataLoader

# Batch size
batch_size = 32

# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

print("DataLoaders created successfully!")
print("Batch size:", batch_size)

DataLoaders created successfully!
Batch size: 32


In [11]:
# Test one batch from the training DataLoader

images, labels = next(iter(train_loader))

print("Batch image shape:", images.shape)
print("Batch labels shape:", labels.shape)
print("Image dtype:", images.dtype)
print("Labels:", labels)

Batch image shape: torch.Size([32, 3, 224, 224])
Batch labels shape: torch.Size([32])
Image dtype: torch.float32
Labels: tensor([0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0,
        0, 0, 0, 1, 0, 0, 1, 1])


In [12]:
# Save the dataset splits for reproducibility

train_df.to_csv("../data/metadata/train_split.csv", index=False)
val_df.to_csv("../data/metadata/val_split.csv", index=False)
test_df.to_csv("../data/metadata/test_split.csv", index=False)

print("Dataset splits saved successfully!")

Dataset splits saved successfully!


In [13]:
# Verify saved split files

import os

split_files = [
    "../data/metadata/train_split.csv",
    "../data/metadata/val_split.csv",
    "../data/metadata/test_split.csv"
]

for file in split_files:
    print(file, "->", os.path.exists(file))

../data/metadata/train_split.csv -> True
../data/metadata/val_split.csv -> True
../data/metadata/test_split.csv -> True
